In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader 
import torchvision.transforms as transforms


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True, download = True,transform= transform)
testset = CIFAR10(root = "./data",train=False, download= True,transform=transform)

C:\Users\dhruv\anaconda3\envs\dl\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True,num_workers=0,pin_memory=True)
testloader = DataLoader(testset,batch_size=64)

In [7]:
class cnn(nn.Module):
    def __init__(self):
        super(cnn,self).__init__()

        self.Conv_layer = nn.Sequential(

            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal_size and stride

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal_size and stride

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) #kernal_size and stride
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.Conv_layer(x)
        x = x.view(x.size(0),-1) # flatterning
        x = self.fc_layers(x)

        return x

In [8]:
# Create the model
model = cnn()

# Move the entire model's parameters to the GPU
model = model.to(device)

# Setup loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [9]:
epochs  = 10;

for epoch in range(epochs):
    model.train()

    epoch_train_loss = 0.0

    for images , lables in trainloader:
        images = images.to(device,non_blocking = True)
        lables = lables.to(device, non_blocking = True)

        # clearing old grad
        optimizer.zero_grad()
        #forward pass
        output = model(images)
        loss = criterion(output,lables)

        #back prop
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()

        
        
    print(f"epoch={epoch+1}/{epochs}, loss={epoch_train_loss/len(trainloader)}")
        
        

        

epoch=1/10, loss=1.3457745867007225
epoch=2/10, loss=0.9075423760334854
epoch=3/10, loss=0.7299849138692822
epoch=4/10, loss=0.606312516857596
epoch=5/10, loss=0.506514613014048
epoch=6/10, loss=0.4123933283454927
epoch=7/10, loss=0.33204730748749145
epoch=8/10, loss=0.2567775391347116
epoch=9/10, loss=0.197820727739607
epoch=10/10, loss=0.15981021074964033


In [14]:
correct_labels =0
total_labels = 0

model.eval()

with torch.no_grad():
    for images , labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted ==labels).sum().item()
        total_labels += labels.size(0)


print(f"accuracy ={(correct_labels/total_labels )*100}")

accuracy =75.53999999999999
